# Signal Detection

Signal detection identifies **out-of-control conditions** in your process using the Western Electric (WECO) rules. These rules detect patterns that indicate special cause variation.

## What You'll Learn

1. Understand all 8 Western Electric rules
2. Configure which rules to apply
3. Interpret signal detection results
4. Visualize rule violations on charts

## Setup

In [ ]:
import numpy as np
import pandas as pd
from processbehavior import ProcessDataFrame
from processbehavior.signals import RuleSet

## Create Data with Various Patterns

We'll create data that exhibits different out-of-control patterns to demonstrate each rule:

In [ ]:
np.random.seed(42)

n = 50
values = np.random.normal(100, 2, n)

# Rule 1: Point beyond limits (index 10)
values[10] = 112  # Beyond 3-sigma

# Rule 4: Run (8 consecutive same side) (indices 20-27)
values[20:28] = np.random.normal(103, 0.5, 8)  # All above center

# Rule 5: Trend (6 consecutive increasing) (indices 35-40)
values[35:41] = [98, 99, 100, 101, 102, 103]

df = pd.DataFrame({
    'day': range(1, n + 1),
    'measurement': np.round(values, 2)
})

print(f"Dataset: {len(df)} observations")
df.head()

## Create Analysis

In [ ]:
pdf = ProcessDataFrame(df)
study = pdf.formulate(
    response=pdf.columns.measurement,
    time=pdf.columns.day
)
result = study.analyze()

## The 8 Western Electric Rules

### Zone Definitions

The rules reference three zones on each side of the centerline:

- **Zone C**: Within 1 sigma of centerline
- **Zone B**: Between 1 and 2 sigma
- **Zone A**: Between 2 and 3 sigma

### The Rules

| Rule | Name | Pattern | Interpretation |
|------|------|---------|----------------|
| 1 | Beyond Limits | 1 point > 3σ from center | Obvious special cause |
| 2 | Zone A | 2 of 3 consecutive in Zone A | Likely shift |
| 3 | Zone B | 4 of 5 consecutive in Zone B+ | Process shifting |
| 4 | Run | 8+ consecutive same side | Sustained shift |
| 5 | Trend | 6+ consecutive increasing/decreasing | Drift |
| 6 | Oscillation | 14+ consecutive alternating | Overcontrol |
| 7 | Hugging Center | 15+ consecutive in Zone C | Reduced variation |
| 8 | Avoiding Center | 8+ consecutive not in Zone C | Bimodal distribution |

## Standard vs. Extended Rules

ProcessBehavior offers three rule sets:

- **`'standard'`**: Rules 1-4 (most common, lower false alarm rate)
- **`'extended'`**: Rules 1-8 (more sensitive, higher false alarm rate)
- **`'all'`**: Same as extended

In [ ]:
# Standard rules (1-4)
signals_std = result.detect_signals(chart='Imr', rules='standard')
print(f"Standard rules: {signals_std.count} signals")

# Extended rules (1-8)
signals_ext = result.detect_signals(chart='Imr', rules='extended')
print(f"Extended rules: {signals_ext.count} signals")

## Examining Signal Results

In [ ]:
signals = result.detect_signals(chart='Imr', rules='extended')

print(f"Has signals: {signals.has_signals}")
print(f"Total count: {signals.count}")
print(f"\nFlagged observations: {signals.flagged_observations}")

In [ ]:
# View all violations
print("All Violations:")
signals.violations

In [ ]:
# Summary by rule
print("\nSummary by Rule:")
signals.summary

In [ ]:
# Violations grouped by rule
print("\nViolations by Rule:")
for rule, violations in signals.by_rule.items():
    print(f"  {rule}: {len(violations)} violation(s)")

## Visualize with Rule Violations

In [ ]:
fig = result.plot(
    show_zones=True,
    show_rules=True,  # Shows all rule violations
    show_signals=True
)
fig.show()

## Custom Rule Configuration

Use the `RuleSet` builder for precise control:

In [ ]:
# Only check for beyond limits and runs
custom_rules = (
    RuleSet()
    .beyond_limits()  # Rule 1
    .run(length=8)    # Rule 4 with 8+ consecutive
    .build()
)

signals_custom = result.detect_signals(chart='Imr', rules=custom_rules)
print(f"Custom rules found: {signals_custom.count} signals")

In [ ]:
# More sensitive trend detection (5 instead of 6)
sensitive_rules = (
    RuleSet()
    .beyond_limits()
    .trend(length=5)  # 5+ consecutive instead of 6
    .build()
)

signals_sensitive = result.detect_signals(chart='Imr', rules=sensitive_rules)
print(f"Sensitive trend detection: {signals_sensitive.count} signals")

## Rule Applicability by Chart Type

Not all rules apply to all chart types:

| Chart Type | Applicable Rules |
|------------|------------------|
| **IMR** | All 8 rules |
| **Xbar** | Rule 1 only |
| **S** | Rule 1 only |
| **R** | Rule 1 only |

### Why the Difference?

- **IMR charts** are time-ordered, so sequential patterns (runs, trends) are meaningful
- **Xbar/S charts** compare subgroups, which may not be time-ordered
- For Xbar/S, only points beyond limits indicate special causes

## Understanding Each Rule

### Rule 1: Beyond Limits

**Pattern**: Single point beyond 3σ limits

**Interpretation**: Almost certainly a special cause. In a stable process, the chance of a point beyond 3σ is about 0.27%.

**Action**: Investigate immediately. What changed?

In [ ]:
# Our data point at index 10 (day 11) should trigger Rule 1
print(f"Value at day 11: {df.loc[10, 'measurement']}")
stats = result.get_statistics('Imr')
print(f"UCL: {stats['ucl']:.2f}")

### Rule 4: Run

**Pattern**: 8+ consecutive points on same side of centerline

**Interpretation**: The process has shifted. Even small shifts (< 1σ) will eventually produce runs.

**Action**: Look for what caused the sustained change.

In [ ]:
# Days 21-28 should all be above centerline
print("Values at days 21-28:")
print(df.loc[20:27, ['day', 'measurement']])
print(f"\nCenterline: {stats['center']:.2f}")

### Rule 5: Trend

**Pattern**: 6+ consecutive points increasing or decreasing

**Interpretation**: Process is drifting. Common causes: tool wear, temperature changes, material degradation.

**Action**: Identify and address the source of drift.

In [ ]:
# Days 36-41 have increasing trend
print("Values at days 36-41:")
print(df.loc[35:40, ['day', 'measurement']])

## False Alarm Rates

More rules = more sensitivity = more false alarms

| Rule Set | Approx. False Alarm Rate |
|----------|-------------------------|
| Rule 1 only | 0.27% per point |
| Rules 1-4 | ~1-2% per point |
| Rules 1-8 | ~3-5% per point |

**Recommendation**: Start with standard rules (1-4). Only use extended rules when you have enough data and can investigate false alarms.

## Summary

In this tutorial, you learned:

- The 8 Western Electric rules detect different patterns
- Use `'standard'` (rules 1-4) for most applications
- Use `'extended'` (rules 1-8) for more sensitive detection
- Use `RuleSet()` builder for custom configurations
- Only Rule 1 applies to Xbar/S charts; all 8 apply to IMR
- More rules = more sensitivity = more false alarms

## Next Steps

- [Western Electric Rules](../reference/weco-rules.md) - Complete rule reference
- [Plotting & Themes](../user-guide/plotting.md) - Visualization options
- [Excel Export](../user-guide/excel-export.md) - Export results